# Persistent Conversation History (DynamoDB)

In `03_converse` the chat history lived in a Python list - it vanished when the program ended. A real chat app needs history that **persists across requests and sessions**, and that's usually per-user. The pattern here stores each message in DynamoDB, keyed by user, so the conversation continues even after a restart.

The slides show this as an AWS Lambda handler (see `lambda_handler.py`). This notebook runs the same logic locally with a fixed `user_id` so you can watch it work.

> **Teaching/Learning Tip:** the model itself is stateless - it has no memory between calls. "Memory" is an illusion *you* create by storing past messages and resending them. DynamoDB is just where we keep them.

## Setup: the DynamoDB table

The table is keyed by `userID` (partition) + `timestamp` (sort), so a query by user returns that user's messages in time order. A `ttl` attribute lets DynamoDB auto-expire old messages.

In [ ]:
import time
from decimal import Decimal
import boto3

REGION = "us-east-1"
TABLE_NAME = "ConversationHistory"
USER_ID = "demo-user"  # stands in for the JWT 'sub' claim in the Lambda version

dynamodb = boto3.resource("dynamodb", region_name=REGION)
ddb_client = boto3.client("dynamodb", region_name=REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

existing = [t.name for t in dynamodb.tables.all()]
if TABLE_NAME in existing:
    table = dynamodb.Table(TABLE_NAME)
    print(f"Using existing table '{TABLE_NAME}'")
else:
    print(f"Creating table '{TABLE_NAME}' (takes ~20s) ...")
    table = dynamodb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[
            {"AttributeName": "userID", "KeyType": "HASH"},
            {"AttributeName": "timestamp", "KeyType": "RANGE"},
        ],
        AttributeDefinitions=[
            {"AttributeName": "userID", "AttributeType": "S"},
            {"AttributeName": "timestamp", "AttributeType": "N"},
        ],
        BillingMode="PAY_PER_REQUEST",
    )
    table.wait_until_exists()
    print("Table ready")

## Slides 4 & 5: store and load helpers

`store_message` writes one message with a 30-day TTL. `get_conversation_history` queries all of a user's messages and rebuilds them into the `messages` shape Converse expects.

In [ ]:
def store_message(user_id, message, role):
    now = Decimal(str(time.time()))
    table.put_item(Item={
        "userID": user_id,
        "timestamp": now,
        "message": message,
        "role": role,
        "ttl": now + (30 * 24 * 60 * 60),  # 30 days
    })


def get_conversation_history(user_id):
    paginator = ddb_client.get_paginator("query")
    pages = paginator.paginate(
        TableName=TABLE_NAME,
        KeyConditionExpression="userID = :val",
        ExpressionAttributeValues={":val": {"S": user_id}},
    )
    messages = []
    for page in pages:
        for item in page.get("Items", []):
            messages.append({
                "role": item["role"]["S"],
                "content": [{"text": item["message"]["S"]}],
            })
    return messages


# Start clean so the demo is repeatable
from boto3.dynamodb.conditions import Key
existing_items = table.query(KeyConditionExpression=Key("userID").eq(USER_ID)).get("Items", [])
with table.batch_writer() as batch:
    for it in existing_items:
        batch.delete_item(Key={"userID": USER_ID, "timestamp": it["timestamp"]})
print(f"Starting fresh. History now: {get_conversation_history(USER_ID)}")

## Slides 1-3: one turn of the conversation

This is what the Lambda does per request: load history, append + store the user message, call Converse with the full history, then store the reply. We'll wrap it in a helper and call it twice.

In [ ]:
def send(user_msg):
    history = get_conversation_history(USER_ID)          # slide 1: load
    history.append({"role": "user", "content": [{"text": user_msg}]})
    store_message(USER_ID, user_msg, "user")             # slide 1: store user msg

    response = bedrock_runtime.converse(                  # slide 2: converse
        modelId="us.amazon.nova-lite-v1:0",
        messages=history,
        system=[{"text": "Please provide a helpful, conversational response "
                         "based on the conversation history."}],
        inferenceConfig={"maxTokens": 300, "temperature": 0.7, "topP": 0.9},
    )
    reply = response["output"]["message"]["content"][0]["text"]
    store_message(USER_ID, reply, "assistant")           # slide 3: store reply
    return reply


print("Turn 1:")
print(send("My name is John and I teach a coding class."))

## The payoff: a second turn remembers the first

The `send` helper reloads history from DynamoDB every call - it keeps no in-memory state. So this second turn only "remembers" because turn 1 was persisted.

In [ ]:
print("Turn 2:")
print(send("What is my name and what do I do?"))

The model answers correctly because the earlier turn was loaded from DynamoDB, not held in memory. In the runnable script `chat_persistent.py`, you can even quit the program entirely, start it again, and it still remembers - that's real persistence.

> **Teaching/Learning Tip:** keying by `userID` gives each user their own private history automatically. In the Lambda version that user ID comes from the verified JWT, so users can never read each other's conversations.

## Cleanup

Delete the table when you're done.

In [ ]:
table.delete()
table.wait_until_not_exists()
print(f"Deleted table '{TABLE_NAME}'.")